# Build Faster with Cortex Code — HOL Setup

This notebook bootstraps the **CardOpsDB** dataset so you can jump straight into the Cortex Code demo (Acts 2-5).

**What it does:**
1. Creates DB, schema, warehouse, and 5 tables
2. Seeds static reference data (Accounts + Merchants)
3. Inserts base transactional rows + generates synthetic data at scale
4. Sets up External Access Integration for dbt package downloads

**Runtime:** ~2 minutes

In [ ]:
from snowflake.snowpark import Session

TARGET_DB = "CDC_TARGET"
TARGET_SCHEMA = "CARDOPS"
TARGET_WH = "CDC_WH"

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    session = Session.builder.configs({"connection_name": "default"}).create()

def run(sql):
    return session.sql(sql).collect()

def seed(table, cols, rows, truncate=True):
    if truncate:
        run(f"TRUNCATE TABLE IF EXISTS {table}")
    col_str = ",".join(f'"{c}"' for c in cols)
    for i in range(0, len(rows), 500):
        batch = rows[i:i+500]
        vals = ",".join(
            "(" + ",".join(
                "NULL" if v is None
                else f"'{v}'" if isinstance(v, (str, bool))
                else str(v)
                for v in r
            ) + ")"
            for r in batch
        )
        run(f"INSERT INTO {table} ({col_str}) VALUES {vals}")
    return len(rows)

run(f"USE ROLE SYSADMIN")
run(f"CREATE DATABASE IF NOT EXISTS {TARGET_DB}")
run(f"CREATE SCHEMA IF NOT EXISTS {TARGET_DB}.{TARGET_SCHEMA}")
run(f"""CREATE WAREHOUSE IF NOT EXISTS {TARGET_WH}
    WAREHOUSE_SIZE='XSMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE""")
run(f"USE DATABASE {TARGET_DB}")
run(f"USE SCHEMA {TARGET_SCHEMA}")
run(f"USE WAREHOUSE {TARGET_WH}")

for ddl in [
    """CREATE TABLE IF NOT EXISTS ACCOUNTS (
        "AccountID" NUMBER(38,0) NOT NULL, "AccountNumber" VARCHAR, "AccountStatus" VARCHAR,
        "CreditLimit" NUMBER(12,2), "CurrentBalance" NUMBER(12,2), "AvailableCredit" NUMBER(12,2),
        "OpenDate" DATE, "Region" VARCHAR, "RiskTier" VARCHAR,
        "CreatedAt" TIMESTAMP_NTZ, "UpdatedAt" TIMESTAMP_NTZ,
        _SNOWFLAKE_INSERTED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_UPDATED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_DELETED BOOLEAN DEFAULT FALSE)""",
    """CREATE TABLE IF NOT EXISTS MERCHANTS (
        "MerchantID" NUMBER(38,0) NOT NULL, "MerchantName" VARCHAR, "CategoryCode" VARCHAR,
        "CategoryName" VARCHAR, "City" VARCHAR, "State" VARCHAR, "ZipCode" VARCHAR, "Country" VARCHAR,
        "CreatedAt" TIMESTAMP_NTZ, "UpdatedAt" TIMESTAMP_NTZ,
        _SNOWFLAKE_INSERTED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_UPDATED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_DELETED BOOLEAN DEFAULT FALSE)""",
    """CREATE TABLE IF NOT EXISTS TRANSACTIONS (
        "TransactionID" NUMBER(38,0) NOT NULL, "AccountID" NUMBER(38,0), "MerchantID" NUMBER(38,0),
        "AuthDateTime" TIMESTAMP_NTZ, "Amount" NUMBER(12,2), "Currency" VARCHAR,
        "AuthDecision" VARCHAR, "DeclineReason" VARCHAR, "CardPresent" BOOLEAN,
        "Channel" VARCHAR, "ResponseCode" VARCHAR, "CreatedAt" TIMESTAMP_NTZ,
        _SNOWFLAKE_INSERTED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_UPDATED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_DELETED BOOLEAN DEFAULT FALSE)""",
    """CREATE TABLE IF NOT EXISTS DISPUTES (
        "DisputeID" NUMBER(38,0) NOT NULL, "AccountID" NUMBER(38,0), "TransactionID" NUMBER(38,0),
        "DisputeAmount" NUMBER(12,2), "DisputeReason" VARCHAR, "Status" VARCHAR,
        "FraudIndicator" BOOLEAN, "MerchantName" VARCHAR,
        "OpenedAt" TIMESTAMP_NTZ, "LastUpdatedAt" TIMESTAMP_NTZ, "ResolvedAt" TIMESTAMP_NTZ,
        _SNOWFLAKE_INSERTED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_UPDATED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_DELETED BOOLEAN DEFAULT FALSE)""",
    """CREATE TABLE IF NOT EXISTS FRAUDALERTS (
        "AlertID" NUMBER(38,0) NOT NULL, "TransactionID" NUMBER(38,0), "AccountID" NUMBER(38,0),
        "AlertScore" NUMBER(5,2), "AlertDecision" VARCHAR, "RuleTriggered" VARCHAR,
        "DeviceID" VARCHAR, "IPAddress" VARCHAR, "CreatedAt" TIMESTAMP_NTZ,
        _SNOWFLAKE_INSERTED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_UPDATED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
        _SNOWFLAKE_DELETED BOOLEAN DEFAULT FALSE)"""
]:
    run(ddl)

print(f"✓ {TARGET_DB}.{TARGET_SCHEMA} ready on {TARGET_WH} — 5 tables created")

In [ ]:
ACCT_COLS = ["AccountID","AccountNumber","AccountStatus","CreditLimit","CurrentBalance","AvailableCredit","OpenDate","Region","RiskTier","CreatedAt","UpdatedAt"]
ACCT_DATA = [
    (1,"****1001","ACTIVE",10000.00,7057.13,2942.87,"2023-01-15","West","HIGH","2026-03-24 18:19:57.700","2026-03-24 18:19:57.700"),
    (2,"****1002","ACTIVE",25000.00,8236.03,16763.97,"2022-06-20","Southeast","LOW","2026-03-24 18:19:57.730","2026-03-24 18:19:57.730"),
    (3,"****1003","ACTIVE",5000.00,1627.31,3372.69,"2024-03-10","West","MEDIUM","2026-03-24 18:19:57.760","2026-03-24 18:19:57.760"),
    (4,"****1004","ACTIVE",15000.00,3726.17,11273.83,"2023-08-01","Southeast","PREMIUM","2026-03-24 18:19:57.850","2026-03-24 18:19:57.850"),
    (5,"****1005","SUSPENDED",8000.00,7890.00,110.00,"2022-11-15","Southwest","HIGH","2026-03-24 18:19:57.876","2026-03-24 18:19:57.876"),
    (6,"****1006","ACTIVE",50000.00,15055.11,34944.89,"2021-04-22","West","PREMIUM","2026-03-24 18:19:57.906","2026-03-24 18:19:57.906"),
    (7,"****1007","ACTIVE",3000.00,1398.98,1601.02,"2024-07-01","Southeast","PREMIUM","2026-03-24 18:19:57.936","2026-03-24 18:19:57.936"),
    (8,"****1008","ACTIVE",20000.00,8359.13,11640.87,"2023-02-28","West","PREMIUM","2026-03-24 18:19:57.963","2026-03-24 18:19:57.963"),
    (9,"****1009","ACTIVE",12000.00,11237.35,762.65,"2022-09-15","Midwest","HIGH","2026-03-24 18:19:57.990","2026-03-24 18:19:57.990"),
    (10,"****1010","ACTIVE",35000.00,5161.98,29838.02,"2021-12-01","Southwest","PREMIUM","2026-03-24 18:19:58.020","2026-03-24 18:19:58.020"),
    (11,"****1011","ACTIVE",7500.00,3890.93,3609.07,"2023-05-10","Midwest","LOW","2026-03-24 18:19:58.050","2026-03-24 18:19:58.050"),
    (12,"****1012","ACTIVE",15000.00,8920.51,6079.49,"2022-08-20","Northeast","LOW","2026-03-24 18:19:58.080","2026-03-24 18:19:58.080"),
    (13,"****1013","CLOSED",10000.00,0.00,0.00,"2021-03-15","Southeast","LOW","2026-03-24 18:19:58.203","2026-03-24 18:19:58.203"),
    (14,"****1014","ACTIVE",22000.00,596.28,21403.72,"2023-11-01","Northeast","LOW","2026-03-24 18:19:58.236","2026-03-24 18:19:58.236"),
    (15,"****1015","ACTIVE",6000.00,3779.28,2220.72,"2024-01-20","Southwest","MEDIUM","2026-03-24 18:19:58.266","2026-03-24 18:19:58.266"),
    (16,"****1016","ACTIVE",40000.00,15287.76,24712.24,"2021-07-10","Northeast","LOW","2026-03-24 18:19:58.296","2026-03-24 18:19:58.296"),
    (17,"****1017","ACTIVE",5000.00,1581.40,3418.60,"2024-06-15","Northeast","MEDIUM","2026-03-24 18:19:58.326","2026-03-24 18:19:58.326"),
    (18,"****1018","ACTIVE",18000.00,9195.85,8804.15,"2023-04-01","Midwest","HIGH","2026-03-24 18:19:58.356","2026-03-24 18:19:58.356"),
    (19,"****1019","SUSPENDED",9000.00,8900.00,100.00,"2022-10-25","Southwest","HIGH","2026-03-24 18:19:58.406","2026-03-24 18:19:58.406"),
    (20,"****1020","ACTIVE",30000.00,7984.58,22015.42,"2022-01-15","Northeast","LOW","2026-03-24 18:19:58.436","2026-03-24 18:19:58.436"),
    (21,"****1021","ACTIVE",11000.00,2954.54,8045.46,"2023-09-05","West","MEDIUM","2026-03-24 18:19:58.466","2026-03-24 18:19:58.466"),
    (22,"****1022","ACTIVE",8000.00,1385.83,6614.17,"2024-02-14","West","HIGH","2026-03-24 18:19:58.496","2026-03-24 18:19:58.496"),
    (23,"****1023","ACTIVE",16000.00,7614.44,8385.56,"2022-12-10","Southeast","HIGH","2026-03-24 18:19:58.526","2026-03-24 18:19:58.526"),
    (24,"****1024","ACTIVE",45000.00,19251.43,25748.57,"2021-06-01","Southwest","MEDIUM","2026-03-24 18:19:58.556","2026-03-24 18:19:58.556"),
    (25,"****1025","ACTIVE",4000.00,342.68,3657.32,"2024-05-20","Southwest","HIGH","2026-03-24 18:19:58.586","2026-03-24 18:19:58.586"),
    (26,"****1026","ACTIVE",13000.00,4577.54,8422.46,"2023-07-15","Northeast","HIGH","2026-03-24 18:19:58.616","2026-03-24 18:19:58.616"),
    (27,"****1027","ACTIVE",7000.00,4166.02,2833.98,"2023-01-30","Northeast","PREMIUM","2026-03-24 18:19:58.646","2026-03-24 18:19:58.646"),
    (28,"****1028","ACTIVE",25000.00,11133.39,13866.61,"2022-04-18","Southwest","HIGH","2026-03-24 18:19:58.676","2026-03-24 18:19:58.676"),
    (29,"****1029","ACTIVE",9500.00,3207.97,6292.03,"2024-04-01","West","HIGH","2026-03-24 18:19:58.706","2026-03-24 18:19:58.706"),
    (30,"****1030","ACTIVE",20000.00,9452.13,10547.87,"2022-07-22","West","MEDIUM","2026-03-24 18:19:58.736","2026-03-24 18:19:58.736"),
    (31,"****1031","ACTIVE",6500.00,777.80,5722.20,"2023-10-12","Northeast","PREMIUM","2026-03-24 18:19:58.766","2026-03-24 18:19:58.766"),
    (32,"****1032","ACTIVE",35000.00,14993.38,20006.62,"2021-09-05","Northeast","MEDIUM","2026-03-24 18:19:58.796","2026-03-24 18:19:58.796"),
    (33,"****1033","ACTIVE",5500.00,1791.13,3708.87,"2024-08-01","Southeast","MEDIUM","2026-03-24 18:19:58.826","2026-03-24 18:19:58.826"),
    (34,"****1034","ACTIVE",12000.00,2787.31,9212.69,"2023-03-20","Southeast","PREMIUM","2026-03-24 18:19:58.856","2026-03-24 18:19:58.856"),
    (35,"****1035","ACTIVE",8500.00,7547.37,952.63,"2022-05-30","Southeast","PREMIUM","2026-03-24 18:19:58.886","2026-03-24 18:19:58.886"),
    (36,"****1036","ACTIVE",28000.00,9798.05,18201.95,"2022-02-14","Southwest","HIGH","2026-03-24 18:19:58.916","2026-03-24 18:19:58.916"),
    (37,"****1037","ACTIVE",4500.00,1116.14,3383.86,"2024-09-10","Southwest","HIGH","2026-03-24 18:19:58.946","2026-03-24 18:19:58.946"),
    (38,"****1038","ACTIVE",17000.00,6547.15,10452.85,"2023-06-25","Northeast","LOW","2026-03-24 18:19:58.976","2026-03-24 18:19:58.976"),
    (39,"****1039","ACTIVE",10000.00,10107.86,0.00,"2022-11-08","Southeast","PREMIUM","2026-03-24 18:19:59.006","2026-03-24 18:19:59.006"),
    (40,"****1040","ACTIVE",42000.00,16496.53,25503.47,"2021-10-15","West","MEDIUM","2026-03-24 18:19:59.036","2026-03-24 18:19:59.036"),
    (41,"****1041","ACTIVE",7500.00,4283.98,3216.02,"2023-12-01","Southeast","MEDIUM","2026-03-24 18:19:59.066","2026-03-24 18:19:59.066"),
    (42,"****1042","ACTIVE",14000.00,6924.90,7075.10,"2023-02-10","West","HIGH","2026-03-24 18:19:59.096","2026-03-24 18:19:59.096"),
    (43,"****1043","ACTIVE",6000.00,5231.94,768.06,"2024-03-15","Midwest","PREMIUM","2026-03-24 18:19:59.126","2026-03-24 18:19:59.126"),
    (44,"****1044","ACTIVE",22000.00,6379.27,15620.73,"2022-08-05","Midwest","PREMIUM","2026-03-24 18:19:59.156","2026-03-24 18:19:59.156"),
    (45,"****1045","ACTIVE",9000.00,2925.88,6074.12,"2023-04-20","Southeast","PREMIUM","2026-03-24 18:19:59.186","2026-03-24 18:19:59.186"),
    (46,"****1046","ACTIVE",32000.00,9856.95,22143.05,"2021-11-20","Midwest","HIGH","2026-03-24 18:19:59.216","2026-03-24 18:19:59.216"),
    (47,"****1047","ACTIVE",5000.00,2968.78,2031.22,"2024-07-12","Southwest","LOW","2026-03-24 18:19:59.246","2026-03-24 18:19:59.246"),
    (48,"****1048","ACTIVE",16000.00,9642.90,6357.10,"2023-08-30","Southeast","LOW","2026-03-24 18:19:59.276","2026-03-24 18:19:59.276"),
    (49,"****1049","ACTIVE",11000.00,8594.86,2405.14,"2022-06-14","Southeast","PREMIUM","2026-03-24 18:19:59.306","2026-03-24 18:19:59.306"),
    (50,"****1050","ACTIVE",38000.00,17333.36,20666.64,"2021-05-01","Midwest","LOW","2026-03-24 18:19:59.336","2026-03-24 18:19:59.336"),
]

MERCH_COLS = ["MerchantID","MerchantName","CategoryCode","CategoryName","City","State","ZipCode","Country","CreatedAt","UpdatedAt"]
MERCH_DATA = [
    (1,"FreshMart Grocery","5411","Groceries","Andersonview","MT","10001","USA","2026-03-24 18:19:59.370","2026-03-24 18:19:59.370"),
    (2,"QuickStop Gas","5541","Gas Stations","New Sammy","LA","90001","USA","2026-03-24 18:19:59.400","2026-03-24 18:19:59.400"),
    (3,"Bella Italia Restaurant","5812","Restaurants","Fort Dewayne","KY","60601","USA","2026-03-24 18:19:59.430","2026-03-24 18:19:59.430"),
    (4,"TechZone Electronics","5732","Electronics","Port Demetrius","HI","94102","USA","2026-03-24 18:19:59.460","2026-03-24 18:19:59.460"),
    (5,"CloudShop Online","5999","Online Retail","Bridgettechester","NC","98101","USA","2026-03-24 18:19:59.490","2026-03-24 18:19:59.490"),
    (6,"SkyHigh Airlines","4511","Airlines","Darefurt","TN","75201","USA","2026-03-24 18:19:59.520","2026-03-24 18:19:59.520"),
    (7,"Grand Plaza Hotel","7011","Hotels","Beavercreek","AZ","33101","USA","2026-03-24 18:19:59.550","2026-03-24 18:19:59.550"),
    (8,"MedPlus Pharmacy","5912","Pharmacies","New Garrettstad","CT","02101","USA","2026-03-24 18:19:59.580","2026-03-24 18:19:59.580"),
    (9,"AutoCare Services","7538","Auto Services","Port Hailey","KY","48201","USA","2026-03-24 18:19:59.610","2026-03-24 18:19:59.610"),
    (10,"StreamFlix Digital","5815","Digital Services","East Gianniburgh","NV","73301","USA","2026-03-24 18:19:59.640","2026-03-24 18:19:59.640"),
    (11,"Sunrise Coffee Co","5814","Coffee Shops","Maryjanehaven","NH","97201","USA","2026-03-24 18:19:59.820","2026-03-24 18:19:59.820"),
    (12,"Metro Department Store","5311","Department Stores","Margarettastad","WY","19101","USA","2026-03-24 18:19:59.850","2026-03-24 18:19:59.850"),
    (13,"GreenLeaf Market","5411","Groceries","New Matilde","VA","80201","USA","2026-03-24 18:19:59.880","2026-03-24 18:19:59.880"),
    (14,"Velocity Fitness","7941","Fitness Centers","Grahamcester","NY","85001","USA","2026-03-24 18:19:59.910","2026-03-24 18:19:59.910"),
    (15,"BookNest Marketplace","5942","Book Stores","Port Elbert","WI","37201","USA","2026-03-24 18:19:59.940","2026-03-24 18:19:59.940"),
    (16,"PetCare Plus","5995","Pet Stores","New Angiechester","DE","30301","USA","2026-03-24 18:19:59.980","2026-03-24 18:19:59.980"),
    (17,"Harbor Seafood Grill","5812","Restaurants","Fort Jaquelin","CO","92101","USA","2026-03-24 18:20:00.010","2026-03-24 18:20:00.010"),
    (18,"NexGen Software","5734","Software","North Brookshaven","NH","98052","USA","2026-03-24 18:20:00.040","2026-03-24 18:20:00.040"),
    (19,"CityPark Parking","7523","Parking","Atlanta","TN","10013","USA","2026-03-24 18:20:00.070","2026-03-24 18:20:00.070"),
    (20,"BrightSmile Dental","8021","Dental Services","Fort Enos","LA","28201","USA","2026-03-24 18:20:00.100","2026-03-24 18:20:00.100"),
    (21,"FuelUp Express","5541","Gas Stations","Fort Jaydeborough","ME","77001","USA","2026-03-24 18:20:00.130","2026-03-24 18:20:00.130"),
    (22,"Wanderlust Travel","4722","Travel Agencies","Tremblayside","ND","89101","USA","2026-03-24 18:20:00.160","2026-03-24 18:20:00.160"),
    (23,"HomeStyle Furniture","5712","Furniture","Bartellshire","IL","55401","USA","2026-03-24 18:20:00.253","2026-03-24 18:20:00.253"),
    (24,"SparkJewelers","5944","Jewelry","West Jarrod","SC","78201","USA","2026-03-24 18:20:00.286","2026-03-24 18:20:00.286"),
    (25,"ConnectMobile","4816","Telecom","South Ulicesborough","SD","43201","USA","2026-03-24 18:20:00.323","2026-03-24 18:20:00.323"),
    (26,"FreshBrew Cafe","5814","Coffee Shops","South Macie","TX","84101","USA","2026-03-24 18:20:00.356","2026-03-24 18:20:00.356"),
    (27,"QuickMed Urgent Care","8011","Healthcare","Bechtelarstad","TX","33601","USA","2026-03-24 18:20:00.386","2026-03-24 18:20:00.386"),
    (28,"GameVault Arcade","7993","Entertainment","South Renee","RI","32801","USA","2026-03-24 18:20:00.416","2026-03-24 18:20:00.416"),
    (29,"CleanWave Laundry","7211","Laundry","Shanaview","NM","95814","USA","2026-03-24 18:20:00.446","2026-03-24 18:20:00.446"),
    (30,"ValuMart Discount","5331","Discount Stores","La Mesa","CO","46201","USA","2026-03-24 18:20:00.476","2026-03-24 18:20:00.476"),
]

a = seed(f"{TARGET_DB}.{TARGET_SCHEMA}.ACCOUNTS", ACCT_COLS, ACCT_DATA)
m = seed(f"{TARGET_DB}.{TARGET_SCHEMA}.MERCHANTS", MERCH_COLS, MERCH_DATA)
print(f"✓ Seeded {a} accounts + {m} merchants")

In [ ]:
TXN_COLS = ["TransactionID","AccountID","MerchantID","AuthDateTime","Amount","Currency","AuthDecision","DeclineReason","CardPresent","Channel","ResponseCode","CreatedAt"]
TXN_DATA = [
    (1,1,1,"2026-03-24 18:20:00.556",2405.73,"USD","DECLINED","CARD_EXPIRED","false","ONLINE","05","2026-03-24 18:20:00.556"),
    (2,2,2,"2026-03-24 18:20:00.593",291.56,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:00.593"),
    (3,3,3,"2026-03-24 18:20:00.640",4693.99,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:00.640"),
    (4,4,4,"2026-03-24 18:20:00.683",2900.73,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:00.683"),
    (5,5,5,"2026-03-24 18:20:00.716",726.19,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:00.716"),
    (6,6,6,"2026-03-24 18:20:00.750",2842.24,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:00.750"),
    (7,7,7,"2026-03-24 18:20:00.786",4674.36,"USD","APPROVED",None,"true","ATM","00","2026-03-24 18:20:00.786"),
    (8,8,8,"2026-03-24 18:20:00.830",1443.60,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:00.830"),
    (9,9,9,"2026-03-24 18:20:00.866",3723.14,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:00.866"),
    (10,10,10,"2026-03-24 18:20:00.893",3246.58,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:00.893"),
    (11,11,11,"2026-03-24 18:20:00.926",2043.28,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:00.926"),
    (12,12,12,"2026-03-24 18:20:00.960",1200.22,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:00.960"),
    (13,13,13,"2026-03-24 18:20:00.990",2389.82,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:00.990"),
    (14,14,14,"2026-03-24 18:20:01.030",3700.63,"USD","APPROVED",None,"true","ATM","00","2026-03-24 18:20:01.030"),
    (15,15,15,"2026-03-24 18:20:01.073",627.68,"USD","DECLINED","INSUFFICIENT_FUNDS","true","POS","05","2026-03-24 18:20:01.073"),
    (16,16,16,"2026-03-24 18:20:01.103",4301.85,"USD","DECLINED","ADDRESS_MISMATCH","true","POS","05","2026-03-24 18:20:01.103"),
    (17,17,17,"2026-03-24 18:20:01.136",2750.52,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:01.136"),
    (18,18,18,"2026-03-24 18:20:01.166",3951.64,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:01.166"),
    (19,19,19,"2026-03-24 18:20:01.200",789.71,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:01.200"),
    (20,20,20,"2026-03-24 18:20:01.230",2857.57,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:01.230"),
    (21,21,21,"2026-03-24 18:20:01.263",3634.15,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:01.263"),
    (22,22,22,"2026-03-24 18:20:01.296",116.21,"USD","APPROVED",None,"true","ATM","00","2026-03-24 18:20:01.296"),
    (23,23,23,"2026-03-24 18:20:01.330",4105.41,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:01.330"),
    (24,24,24,"2026-03-24 18:20:01.370",561.40,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:01.370"),
    (25,25,25,"2026-03-24 18:20:01.403",1810.45,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:01.403"),
    (26,26,26,"2026-03-24 18:20:01.446",254.57,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:01.446"),
    (27,27,27,"2026-03-24 18:20:01.480",2067.66,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:01.480"),
    (28,28,28,"2026-03-24 18:20:01.510",3822.32,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:01.510"),
    (29,29,29,"2026-03-24 18:20:01.543",754.41,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:01.543"),
    (30,30,30,"2026-03-24 18:20:01.573",3401.37,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:01.573"),
    (31,31,1,"2026-03-24 18:20:01.606",4086.59,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:01.606"),
    (32,32,2,"2026-03-24 18:20:01.646",4519.08,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:01.646"),
    (33,33,3,"2026-03-24 18:20:01.680",2307.83,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:01.680"),
    (34,34,4,"2026-03-24 18:20:01.710",975.61,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:01.710"),
    (35,35,5,"2026-03-24 18:20:01.740",2770.57,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:01.740"),
    (36,36,6,"2026-03-24 18:20:01.773",2433.81,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:01.773"),
    (37,37,7,"2026-03-24 18:20:01.803",1243.20,"USD","DECLINED","ADDRESS_MISMATCH","false","ONLINE","05","2026-03-24 18:20:01.803"),
    (38,38,8,"2026-03-24 18:20:01.833",1902.69,"USD","DECLINED","INSUFFICIENT_FUNDS","false","MOBILE","05","2026-03-24 18:20:01.833"),
    (39,39,9,"2026-03-24 18:20:01.866",1788.26,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:01.866"),
    (40,40,10,"2026-03-24 18:20:01.900",3915.28,"USD","DECLINED","INSUFFICIENT_FUNDS","true","POS","05","2026-03-24 18:20:01.900"),
    (41,41,11,"2026-03-24 18:20:01.930",1312.88,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:01.930"),
    (42,42,12,"2026-03-24 18:20:01.963",3736.87,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:01.963"),
    (43,43,13,"2026-03-24 18:20:01.993",2210.06,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:01.993"),
    (44,44,14,"2026-03-24 18:20:02.026",698.54,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:02.026"),
    (45,45,15,"2026-03-24 18:20:02.066",2830.72,"USD","APPROVED",None,"true","ATM","00","2026-03-24 18:20:02.066"),
    (46,46,16,"2026-03-24 18:20:02.096",4077.80,"USD","DECLINED","VELOCITY_LIMIT","true","POS","05","2026-03-24 18:20:02.096"),
    (47,47,17,"2026-03-24 18:20:02.126",4216.95,"USD","APPROVED",None,"true","ATM","00","2026-03-24 18:20:02.126"),
    (48,48,18,"2026-03-24 18:20:02.170",65.25,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.170"),
    (49,49,19,"2026-03-24 18:20:02.203",3518.95,"USD","DECLINED","ADDRESS_MISMATCH","true","POS","05","2026-03-24 18:20:02.203"),
    (50,50,20,"2026-03-24 18:20:02.233",4385.18,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:02.233"),
    (51,1,21,"2026-03-24 18:20:02.266",4840.43,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:02.266"),
    (52,2,22,"2026-03-24 18:20:02.296",2362.31,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.296"),
    (53,3,23,"2026-03-24 18:20:02.333",4698.18,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:02.333"),
    (54,4,24,"2026-03-24 18:20:02.366",322.95,"USD","DECLINED","CVV_MISMATCH","true","POS","05","2026-03-24 18:20:02.366"),
    (55,5,25,"2026-03-24 18:20:02.406",111.55,"USD","DECLINED","CARD_EXPIRED","true","POS","05","2026-03-24 18:20:02.406"),
    (56,6,26,"2026-03-24 18:20:02.446",4503.41,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.446"),
    (57,7,27,"2026-03-24 18:20:02.473",4846.04,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:02.473"),
    (58,8,28,"2026-03-24 18:20:02.510",2616.06,"USD","DECLINED","INSUFFICIENT_FUNDS","false","MOBILE","05","2026-03-24 18:20:02.510"),
    (59,9,29,"2026-03-24 18:20:02.540",3945.83,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.540"),
    (60,10,30,"2026-03-24 18:20:02.570",180.97,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:02.570"),
    (61,11,1,"2026-03-24 18:20:02.616",825.01,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:02.616"),
    (62,12,2,"2026-03-24 18:20:02.653",4526.10,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.653"),
    (63,13,3,"2026-03-24 18:20:02.686",1243.98,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.686"),
    (64,14,4,"2026-03-24 18:20:02.713",4914.04,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.713"),
    (65,15,5,"2026-03-24 18:20:02.746",2814.01,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:02.746"),
    (66,16,6,"2026-03-24 18:20:02.780",4448.28,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.780"),
    (67,17,7,"2026-03-24 18:20:02.810",1180.93,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:02.810"),
    (68,18,8,"2026-03-24 18:20:02.843",3723.17,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:02.843"),
    (69,19,9,"2026-03-24 18:20:02.873",166.99,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:02.873"),
    (70,20,10,"2026-03-24 18:20:02.906",639.93,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.906"),
    (71,21,11,"2026-03-24 18:20:02.940",4219.29,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:02.940"),
    (72,22,12,"2026-03-24 18:20:02.970",2181.02,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:02.970"),
    (73,23,13,"2026-03-24 18:20:03.003",4941.48,"USD","APPROVED",None,"true","ATM","00","2026-03-24 18:20:03.003"),
    (74,24,14,"2026-03-24 18:20:03.033",2380.64,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.033"),
    (75,25,15,"2026-03-24 18:20:03.070",834.91,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:03.070"),
    (76,26,16,"2026-03-24 18:20:03.103",2873.42,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:03.103"),
    (77,27,17,"2026-03-24 18:20:03.133",3733.05,"USD","FLAGGED",None,"false","ONLINE","01","2026-03-24 18:20:03.133"),
    (78,28,18,"2026-03-24 18:20:03.166",4658.77,"USD","DECLINED","VELOCITY_LIMIT","true","ATM","05","2026-03-24 18:20:03.166"),
    (79,29,19,"2026-03-24 18:20:03.200",4658.15,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:03.200"),
    (80,30,20,"2026-03-24 18:20:03.240",1299.31,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.240"),
    (81,31,21,"2026-03-24 18:20:03.270",4588.45,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.270"),
    (82,32,22,"2026-03-24 18:20:03.303",1533.93,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.303"),
    (83,33,23,"2026-03-24 18:20:03.336",1763.29,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.336"),
    (84,34,24,"2026-03-24 18:20:03.366",4282.44,"USD","FLAGGED",None,"false","ONLINE","01","2026-03-24 18:20:03.366"),
    (85,35,25,"2026-03-24 18:20:03.400",2412.91,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:03.400"),
    (86,36,26,"2026-03-24 18:20:03.436",798.01,"USD","DECLINED","VELOCITY_LIMIT","false","ONLINE","05","2026-03-24 18:20:03.436"),
    (87,37,27,"2026-03-24 18:20:03.476",178.75,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:03.476"),
    (88,38,28,"2026-03-24 18:20:03.513",4485.49,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.513"),
    (89,39,29,"2026-03-24 18:20:03.543",3611.28,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:03.543"),
    (90,40,30,"2026-03-24 18:20:03.573",1756.47,"USD","APPROVED",None,"true","POS","00","2026-03-24 18:20:03.573"),
    (91,41,1,"2026-03-24 18:20:03.603",2643.37,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:03.603"),
    (92,42,2,"2026-03-24 18:20:03.630",3882.88,"USD","APPROVED",None,"true","ATM","00","2026-03-24 18:20:03.630"),
    (93,43,3,"2026-03-24 18:20:03.660",3736.35,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.660"),
    (94,44,4,"2026-03-24 18:20:03.696",226.30,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.696"),
    (95,45,5,"2026-03-24 18:20:03.730",650.64,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:03.730"),
    (96,46,6,"2026-03-24 18:20:03.763",4316.83,"USD","APPROVED",None,"false","MOBILE","00","2026-03-24 18:20:03.763"),
    (97,47,7,"2026-03-24 18:20:03.793",3766.91,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.793"),
    (98,48,8,"2026-03-24 18:20:03.826",1545.44,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.826"),
    (99,49,9,"2026-03-24 18:20:03.863",1722.96,"USD","APPROVED",None,"false","ONLINE","00","2026-03-24 18:20:03.863"),
    (100,50,10,"2026-03-24 18:20:03.893",2972.43,"USD","FLAGGED",None,"true","POS","01","2026-03-24 18:20:03.893"),
]

DISPUTE_COLS = ["DisputeID","AccountID","TransactionID","DisputeAmount","DisputeReason","Status","FraudIndicator","MerchantName","OpenedAt","LastUpdatedAt","ResolvedAt"]
DISPUTE_DATA = [
    (1,1,1,2405.73,"WRONG_AMOUNT","DENIED","false","FreshMart Grocery","2026-03-24 18:20:17.910","2026-03-24 19:04:07.333","2026-03-24 19:04:07.333"),
    (2,18,18,3951.64,"WRONG_AMOUNT","DENIED","true","NexGen Software","2026-03-24 18:20:18.276","2026-03-24 19:04:18.560","2026-03-24 19:04:18.560"),
    (3,35,35,2770.57,"UNAUTHORIZED","RESOLVED","false","CloudShop Online","2026-03-24 18:20:18.430","2026-03-24 19:07:03.513","2026-03-24 19:07:03.513"),
    (4,2,52,2362.31,"UNAUTHORIZED","RESOLVED","false","Wanderlust Travel","2026-03-24 18:20:18.626","2026-03-24 18:20:18.626","2026-03-24 18:20:18.626"),
    (5,19,69,166.99,"NOT_RECEIVED","RESOLVED","false","AutoCare Services","2026-03-24 18:20:18.843","2026-03-24 19:04:15.636","2026-03-24 19:04:15.636"),
    (6,36,86,798.01,"DUPLICATE","RESOLVED","false","FreshBrew Cafe","2026-03-24 18:20:19.090","2026-03-24 19:04:07.063","2026-03-24 19:04:07.063"),
    (7,3,3,3330.91,"NOT_RECEIVED","RESOLVED","false","GreenLeaf Market","2026-03-24 18:20:19.376","2026-03-24 18:20:19.376","2026-03-24 18:20:19.376"),
    (8,20,70,4606.60,"QUALITY_ISSUE","DENIED","false","ValuMart Discount","2026-03-24 18:20:19.566","2026-03-24 19:04:25.540","2026-03-24 19:04:25.540"),
    (9,37,37,2330.79,"WRONG_AMOUNT","DENIED","false","Harbor Seafood Grill","2026-03-24 18:20:19.776","2026-03-24 19:04:07.500","2026-03-24 19:04:07.500"),
    (10,4,4,422.08,"DUPLICATE","DENIED","false","TechZone Electronics","2026-03-24 18:20:19.956","2026-03-24 19:09:12.020","2026-03-24 19:09:12.020"),
    (11,21,71,3516.82,"NOT_RECEIVED","DENIED","false","FuelUp Express","2026-03-24 18:20:20.183","2026-03-24 19:14:21.813","2026-03-24 19:14:21.813"),
    (12,38,38,4798.30,"NOT_RECEIVED","DENIED","true","MedPlus Pharmacy","2026-03-24 18:20:20.340","2026-03-24 19:12:07.450","2026-03-24 19:12:07.450"),
    (13,5,5,2487.84,"NOT_RECEIVED","DENIED","false","ConnectMobile","2026-03-24 18:20:20.486","2026-03-24 18:20:20.486","2026-03-24 18:20:20.486"),
    (14,22,22,3730.62,"UNAUTHORIZED","RESOLVED","true","Metro Department Store","2026-03-24 18:20:20.670","2026-03-24 18:20:20.670","2026-03-24 18:20:20.670"),
    (15,39,39,1013.08,"UNAUTHORIZED","PROVISIONAL_CREDIT","true","CleanWave Laundry","2026-03-24 18:20:20.890","2026-03-24 19:08:25.450",None),
    (16,6,56,3543.79,"DUPLICATE","RESOLVED","false","PetCare Plus","2026-03-24 18:20:21.126","2026-03-24 19:04:13.100","2026-03-24 19:04:13.100"),
    (17,23,23,1979.88,"WRONG_AMOUNT","RESOLVED","false","Bella Italia Restaurant","2026-03-24 18:20:21.270","2026-03-24 19:10:56.466","2026-03-24 19:10:56.466"),
    (18,40,90,3704.83,"DUPLICATE","RESOLVED","false","BrightSmile Dental","2026-03-24 18:20:21.573","2026-03-24 18:20:21.573","2026-03-24 18:20:21.573"),
    (19,7,7,3280.54,"NOT_RECEIVED","DENIED","true","Grand Plaza Hotel","2026-03-24 18:20:21.766","2026-03-24 19:10:32.323","2026-03-24 19:10:32.323"),
    (20,24,24,2353.18,"DUPLICATE","RESOLVED","false","SparkJewelers","2026-03-24 18:20:22.006","2026-03-24 19:09:34.333","2026-03-24 19:09:34.333"),
    (21,41,41,761.09,"UNAUTHORIZED","RESOLVED","false","Sunrise Coffee Co","2026-03-24 18:20:22.210","2026-03-24 19:11:42.123","2026-03-24 19:11:42.123"),
    (22,8,8,4181.16,"WRONG_AMOUNT","RESOLVED","true","GameVault Arcade","2026-03-24 18:20:22.410","2026-03-24 19:14:33.610","2026-03-24 19:14:33.610"),
    (23,25,75,3201.29,"QUALITY_ISSUE","RESOLVED","true","BookNest Marketplace","2026-03-24 18:20:22.626","2026-03-24 18:20:22.626","2026-03-24 18:20:22.626"),
    (24,42,92,523.12,"DUPLICATE","DENIED","false","QuickStop Gas","2026-03-24 18:20:22.770","2026-03-24 19:04:32.263","2026-03-24 19:04:32.263"),
    (25,9,9,4940.77,"DUPLICATE","RESOLVED","false","CityPark Parking","2026-03-24 18:20:22.950","2026-03-24 19:09:53.363","2026-03-24 19:09:53.363"),
]

ALERT_COLS = ["AlertID","TransactionID","AccountID","AlertScore","AlertDecision","RuleTriggered","DeviceID","IPAddress","CreatedAt"]
ALERT_DATA = [
    (1,1,1,74.66,"BLOCK","GEO_ANOMALY","806af815-f087-45f1-9262-f4f8f8a7efeb","73.26.131.87","2026-03-24 18:20:05.000"),
    (2,12,12,51.63,"FLAG","HIGH_RISK_COUNTRY","4cbf165a-138c-4280-9542-9773ab60794a","214.102.107.124","2026-03-24 18:20:05.100"),
    (3,23,23,28.74,"PASS","UNUSUAL_MERCHANT","88b7693d-4f44-4d50-941c-f521b6ce9402","135.133.164.158","2026-03-24 18:20:05.200"),
    (4,34,34,38.39,"FLAG","GEO_ANOMALY","07dd0bbc-fed0-47b5-acec-d7d1d96f03bc","70.170.42.155","2026-03-24 18:20:05.300"),
    (5,45,45,55.40,"FLAG","DEVICE_MISMATCH","43b04e18-4a03-40ae-bd24-3b66b4f362e9","14.178.132.128","2026-03-24 18:20:05.400"),
    (6,56,6,76.84,"BLOCK","VELOCITY","76837684-ab08-4262-b7e0-0dedc97e30ad","28.176.229.165","2026-03-24 18:20:05.500"),
    (7,67,17,18.58,"PASS","AMOUNT_SPIKE","9fabb3cc-0d6f-4b70-8d4c-bf35ce6e4f60","111.188.25.64","2026-03-24 18:20:05.600"),
    (8,78,28,93.77,"BLOCK","AMOUNT_SPIKE","b4dc0523-8177-4251-942b-f54b09f9ccdd","104.244.230.126","2026-03-24 18:20:05.700"),
    (9,89,39,32.45,"FLAG","GEO_ANOMALY","b57f96cb-12ba-4c63-9254-14c7597271ff","65.40.234.54","2026-03-24 18:20:05.800"),
    (10,100,50,80.02,"BLOCK","UNUSUAL_MERCHANT","336fdddd-8af8-4ef3-a6bd-701dcc24ca74","123.232.5.207","2026-03-24 18:20:05.900"),
    (11,11,11,36.83,"FLAG","AMOUNT_SPIKE","72d64331-f16f-4b54-b646-21c70b272042","145.11.95.70","2026-03-24 18:20:06.000"),
    (12,22,22,36.93,"FLAG","GEO_ANOMALY","462af51e-4f1d-4953-adef-110135ec3eb8","24.150.63.119","2026-03-24 18:20:06.100"),
    (13,33,33,36.04,"FLAG","VELOCITY","40fa9f59-bdd0-4389-9a16-97a5a56f7cdf","20.146.213.250","2026-03-24 18:20:06.200"),
    (14,44,44,78.88,"BLOCK","UNUSUAL_MERCHANT","d8dde667-d6c8-4d05-92a4-c0658633e6ae","118.155.130.144","2026-03-24 18:20:06.300"),
    (15,55,5,24.73,"PASS","DEVICE_MISMATCH","6b18e213-26f1-408e-a6e7-583aa441ea89","183.150.204.54","2026-03-24 18:20:06.400"),
    (16,66,16,38.01,"FLAG","DEVICE_MISMATCH","47bce1dc-6a6e-4bd5-8df8-022289d25451","182.233.7.250","2026-03-24 18:20:06.500"),
    (17,77,27,19.05,"PASS","UNUSUAL_MERCHANT","9e2474d4-780d-400c-b801-134d0f048c6c","12.123.168.158","2026-03-24 18:20:06.600"),
    (18,88,38,71.34,"BLOCK","HIGH_RISK_COUNTRY","f1e2fd32-31cf-4679-a19b-8ec68fb410f6","207.122.13.103","2026-03-24 18:20:06.700"),
    (19,99,49,73.08,"BLOCK","GEO_ANOMALY","1d955912-6956-44f7-a761-cbc1778013a2","22.42.246.162","2026-03-24 18:20:06.800"),
    (20,10,10,75.45,"BLOCK","DEVICE_MISMATCH","b27c3d04-8ecb-4e62-b3bf-c6577b0e14ed","35.220.21.22","2026-03-24 18:20:06.900"),
    (21,21,21,76.57,"BLOCK","RAPID_SUCCESSION","9b2c34ba-db61-4dd8-822b-a22c45d211ab","2.173.148.85","2026-03-24 18:20:07.000"),
    (22,32,32,20.64,"PASS","GEO_ANOMALY","4baf4335-0aad-4cc7-b26c-0e3d42de0101","41.137.224.253","2026-03-24 18:20:07.100"),
    (23,43,43,50.82,"FLAG","GEO_ANOMALY","d68c34ef-7619-43b6-951c-75ac9154b79f","106.118.92.224","2026-03-24 18:20:07.200"),
    (24,54,4,52.48,"FLAG","UNUSUAL_MERCHANT","06f4260c-2a59-4465-89b2-78742f413a37","14.128.250.12","2026-03-24 18:20:07.300"),
    (25,65,15,16.09,"PASS","VELOCITY","220ed383-9ea1-4eb9-ba9b-e89fd87aa28c","6.104.122.175","2026-03-24 18:20:07.400"),
    (26,76,26,30.45,"FLAG","RAPID_SUCCESSION","d2ed80ab-3bfb-4f82-8b08-12570d05a3de","68.216.40.23","2026-03-24 18:20:07.500"),
    (27,87,37,11.12,"PASS","RAPID_SUCCESSION","7f4e16a8-bfbc-4b20-bab3-b27a017a646b","200.206.78.195","2026-03-24 18:20:07.600"),
    (28,98,48,29.49,"PASS","GEO_ANOMALY","42ee57ac-d128-4c2d-9b88-4cc1f8f65563","80.18.68.38","2026-03-24 18:20:07.700"),
    (29,9,9,64.45,"FLAG","GEO_ANOMALY","c837743a-aea3-46a7-b184-5d03c23152f3","79.101.132.247","2026-03-24 18:20:07.800"),
    (30,20,20,83.99,"BLOCK","UNUSUAL_MERCHANT","5bf7dbe5-f90e-4387-a26d-91bc41d836b8","62.30.72.252","2026-03-24 18:20:07.900"),
    (31,31,31,93.38,"BLOCK","RAPID_SUCCESSION","1c1e3913-86a1-4e38-b19e-92bd72b4ceec","25.53.165.192","2026-03-24 18:20:08.000"),
    (32,42,42,25.55,"PASS","HIGH_RISK_COUNTRY","6a36a012-75dd-4389-b40c-a40331667142","7.169.167.164","2026-03-24 18:20:08.100"),
    (33,53,3,25.56,"PASS","RAPID_SUCCESSION","e0eb0643-1ae7-4758-9a8f-7acb515435fb","84.170.41.85","2026-03-24 18:20:08.200"),
    (34,64,14,27.13,"PASS","RAPID_SUCCESSION","24579b27-cc13-48cf-be3a-f76c9da8fd7a","136.132.164.48","2026-03-24 18:20:08.300"),
    (35,75,25,78.75,"BLOCK","RAPID_SUCCESSION","a9fb85b3-e596-42fc-ab79-fc68d782dc46","70.27.142.58","2026-03-24 18:20:08.400"),
    (36,86,36,8.08,"PASS","UNUSUAL_MERCHANT","8720f5a8-075c-4aa3-a8f3-21e652c3bdb0","100.85.212.153","2026-03-24 18:20:08.500"),
    (37,97,47,3.24,"PASS","DEVICE_MISMATCH","202b5ff5-8562-4bbf-b445-e9e91a55744a","207.82.106.164","2026-03-24 18:20:08.600"),
    (38,8,8,87.51,"BLOCK","UNUSUAL_MERCHANT","6381e7fc-b3c0-44ee-9578-e4e3bbfabdd6","60.47.197.28","2026-03-24 18:20:08.700"),
    (39,19,19,86.69,"BLOCK","AMOUNT_SPIKE","af6571c2-ecca-4585-8065-89090727221f","35.32.130.209","2026-03-24 18:20:08.800"),
    (40,30,30,69.68,"FLAG","RAPID_SUCCESSION","18c10fda-4747-47fe-8f2e-dd899ecff3b2","17.166.233.68","2026-03-24 18:20:08.900"),
]

t = seed("TRANSACTIONS", TXN_COLS, TXN_DATA)
d = seed("DISPUTES", DISPUTE_COLS, DISPUTE_DATA)
a = seed("FRAUDALERTS", ALERT_COLS, ALERT_DATA)
print(f"✓ Seeded base data: {t} transactions, {d} disputes, {a} fraud alerts")

In [ ]:
import random, uuid
from datetime import datetime, timedelta

account_ids = list(range(1, 51))
merchant_ids = list(range(1, 31))
channels = ["ONLINE", "MOBILE", "POS", "ATM"]
auth_decisions = ["APPROVED"]*7 + ["DECLINED"]*2 + ["FLAGGED"]
decline_reasons = ["INSUFFICIENT_FUNDS", "CARD_EXPIRED", "CVV_MISMATCH", "ADDRESS_MISMATCH", "VELOCITY_LIMIT"]
alert_decisions = ["PASS"]*2 + ["FLAG"]*3 + ["BLOCK"]
rules_triggered = ["AMOUNT_SPIKE", "HIGH_RISK_COUNTRY", "UNUSUAL_MERCHANT", "VELOCITY", "DEVICE_MISMATCH", "GEO_ANOMALY", "RAPID_SUCCESSION"]
dispute_reasons = ["UNAUTHORIZED", "WRONG_AMOUNT", "NOT_RECEIVED", "DUPLICATE", "QUALITY_ISSUE"]
dispute_statuses = ["OPENED", "INVESTIGATING", "PROVISIONAL_CREDIT", "RESOLVED", "DENIED"]

max_txn = run(f'SELECT MAX("TransactionID") FROM {TARGET_DB}.{TARGET_SCHEMA}.TRANSACTIONS')[0][0] or 100
max_disp = run(f'SELECT MAX("DisputeID") FROM {TARGET_DB}.{TARGET_SCHEMA}.DISPUTES')[0][0] or 25
max_alert = run(f'SELECT MAX("AlertID") FROM {TARGET_DB}.{TARGET_SCHEMA}.FRAUDALERTS')[0][0] or 40

merch_map = {r[0]: r[1] for r in run(f'SELECT "MerchantID", "MerchantName" FROM {TARGET_DB}.{TARGET_SCHEMA}.MERCHANTS')}
base_time = datetime(2026, 3, 24, 18, 20, 0)

TARGET_TXN, TARGET_DISPUTES, TARGET_ALERTS = 5000, 1500, 2500

print(f"Generating {TARGET_TXN} transactions...")
txn_rows = []
for i in range(TARGET_TXN):
    dec = random.choice(auth_decisions)
    ts = str(base_time + timedelta(seconds=random.uniform(0, 5400)))
    txn_rows.append((
        max_txn+1+i, random.choice(account_ids), random.choice(merchant_ids),
        ts, round(random.uniform(5, 4999.99), 2), "USD", dec,
        random.choice(decline_reasons) if dec == "DECLINED" else None,
        str(random.choice([True, False])).lower(), random.choice(channels),
        "05" if dec == "DECLINED" else ("01" if dec == "FLAGGED" else "00"), ts
    ))
seed(f"{TARGET_DB}.{TARGET_SCHEMA}.TRANSACTIONS", TXN_COLS, txn_rows, truncate=False)
print(f"✓ {len(txn_rows)} transactions inserted")

all_txn_ids = list(range(1, max_txn+1)) + [r[0] for r in txn_rows]

print(f"Generating {TARGET_DISPUTES} disputes...")
disp_rows = []
for i in range(TARGET_DISPUTES):
    status = random.choice(dispute_statuses)
    opened = base_time + timedelta(seconds=random.uniform(0, 5400))
    updated = opened + timedelta(seconds=random.uniform(0, 3600))
    m = random.choice(merchant_ids)
    disp_rows.append((
        max_disp+1+i, random.choice(account_ids), random.choice(all_txn_ids),
        round(random.uniform(50, 4999.99), 2), random.choice(dispute_reasons), status,
        str(random.choice([True, False])).lower(), merch_map.get(m, "Unknown"),
        str(opened), str(updated), str(updated) if status in ("RESOLVED","DENIED") else None
    ))
seed(f"{TARGET_DB}.{TARGET_SCHEMA}.DISPUTES", DISPUTE_COLS, disp_rows, truncate=False)
print(f"✓ {len(disp_rows)} disputes inserted")

print(f"Generating {TARGET_ALERTS} fraud alerts...")
alert_rows = []
for i in range(TARGET_ALERTS):
    ts = str(base_time + timedelta(seconds=random.uniform(0, 5400)))
    ip = f"{random.randint(1,254)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
    alert_rows.append((
        max_alert+1+i, random.choice(all_txn_ids), random.choice(account_ids),
        round(random.uniform(1, 99.99), 2), random.choice(alert_decisions),
        random.choice(rules_triggered), str(uuid.uuid4()), ip, ts
    ))
seed(f"{TARGET_DB}.{TARGET_SCHEMA}.FRAUDALERTS", ALERT_COLS, alert_rows, truncate=False)
print(f"✓ {len(alert_rows)} fraud alerts inserted")

totals = run(f"""SELECT
    (SELECT COUNT(*) FROM {TARGET_DB}.{TARGET_SCHEMA}.TRANSACTIONS),
    (SELECT COUNT(*) FROM {TARGET_DB}.{TARGET_SCHEMA}.DISPUTES),
    (SELECT COUNT(*) FROM {TARGET_DB}.{TARGET_SCHEMA}.FRAUDALERTS)""")[0]
print(f"\nFinal totals: Transactions={totals[0]}, Disputes={totals[1]}, FraudAlerts={totals[2]}")

In [ ]:
run("USE ROLE SYSADMIN")
run(f"CREATE SCHEMA IF NOT EXISTS {TARGET_DB}.ANALYTICS")
run(f"""CREATE NETWORK RULE IF NOT EXISTS {TARGET_DB}.ANALYTICS.DBT_DEPS_NETWORK_RULE
    MODE = EGRESS TYPE = HOST_PORT
    VALUE_LIST = ('hub.getdbt.com', 'codeload.github.com')""")

run("USE ROLE ACCOUNTADMIN")
run(f"""CREATE EXTERNAL ACCESS INTEGRATION IF NOT EXISTS DBT_DEPS_EXT_ACCESS
    ALLOWED_NETWORK_RULES = ({TARGET_DB}.ANALYTICS.DBT_DEPS_NETWORK_RULE) ENABLED = TRUE""")
run("GRANT USAGE ON INTEGRATION DBT_DEPS_EXT_ACCESS TO ROLE SYSADMIN")
run("USE ROLE SYSADMIN")
print("✓ EAI created — dbt can now download packages from GitHub and dbt Hub")

In [ ]:
rows = run(f"""
    SELECT 'ACCOUNTS' AS t, COUNT(*) FROM {TARGET_DB}.{TARGET_SCHEMA}.ACCOUNTS UNION ALL
    SELECT 'MERCHANTS', COUNT(*) FROM {TARGET_DB}.{TARGET_SCHEMA}.MERCHANTS UNION ALL
    SELECT 'TRANSACTIONS', COUNT(*) FROM {TARGET_DB}.{TARGET_SCHEMA}.TRANSACTIONS UNION ALL
    SELECT 'DISPUTES', COUNT(*) FROM {TARGET_DB}.{TARGET_SCHEMA}.DISPUTES UNION ALL
    SELECT 'FRAUDALERTS', COUNT(*) FROM {TARGET_DB}.{TARGET_SCHEMA}.FRAUDALERTS ORDER BY 1""")
for r in rows:
    print(f"  {r[0]:15s} {r[1]:,} rows")

## Setup Complete

Your CardOpsDB dataset is ready. Expected counts:

| Table | Rows |
|-------|------|
| ACCOUNTS | 50 |
| MERCHANTS | 30 |
| TRANSACTIONS | ~5,100 |
| DISPUTES | ~1,525 |
| FRAUDALERTS | ~2,540 |

**Next steps:**
1. Open **Cortex Code in Snowsight**
2. Feed it the **Act 2 skill** to begin data exploration + dbt build
3. Continue through Acts 3-4 (Semantic View, Cortex Agent)
4. Open **Snowflake Intelligence** for Act 5 (chat with your data)